<a href="https://colab.research.google.com/github/faisalhamza909-blip/mental-health-clustering-case-study-/blob/main/mental_health_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [183]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [184]:
import pandas as pd
path = "/content/drive/MyDrive/case study/mental-heath-in-tech-2016_20161114.csv"
df = pd.read_csv(path)

# Data Exploration
Here we are just exploring the data

In [185]:
df.shape

(1433, 63)

we have 1433 rows and 63 columns


In [186]:
df.head

<bound method NDFrame.head of       Are you self-employed?  \
0                          0   
1                          0   
2                          0   
3                          1   
4                          0   
...                      ...   
1428                       1   
1429                       1   
1430                       0   
1431                       0   
1432                       0   

     How many employees does your company or organization have?  \
0                                                26-100           
1                                                  6-25           
2                                                  6-25           
3                                                   NaN           
4                                                  6-25           
...                                                 ...           
1428                                                NaN           
1429                                                NaN           
1430                                            100-500           
1431                                            100-500           
1432                                            100-500           

      Is your employer primarily a tech company/organization?  \
0                                                   1.0         
1                                                   1.0         
2                                                   1.0         
3                                                   NaN         
4                                                   0.0         
...                                                 ...         
1428                                                NaN         
1429                                                NaN         
1430                                                1.0         
1431                                                0.0         
1432                                                1.0         

      Is your primary role within your company related to tech/IT?  \
0                                                   NaN              
1                                                   NaN              
2                                                   NaN              
3                                                   NaN              
4                                                   1.0              
...                                                 ...              
1428                                                NaN              
1429                                                NaN              
1430                                                NaN              
1431                                                1.0              
1432                                                NaN              

     Does your employer provide mental health benefits as part of healthcare coverage?  \
0                       Not eligible for coverage / N/A                                  
1                                                    No                                  
2                                                    No                                  
3                                                   NaN                                  
4                                                   Yes                                  
...                                                 ...                                  
1428                                                NaN                                  
1429                                                NaN                                  
1430                                                Yes                                  
1431                                       I don't know                                  
1432                                                Yes                                  

     Do you know the options for mental health care available under your empl

In [187]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 63 columns):
 #   Column                                                                                                                                                                            Non-Null Count  Dtype  
---  ------                                                                                                                                                                            --------------  -----  
 0   Are you self-employed?                                                                                                                                                            1433 non-null   int64  
 1   How many employees does your company or organization have?                                                                                                                        1146 non-null   object 
 2   Is your employer primarily a tech company/organization?     

Now we need to calculate the percentage of missing answers for each survey to see which column may neeed to be dropped or handled specially

In [188]:
missing_values = df.isnull()
mean_of_missing_values = missing_values.mean()
missing_percentage = mean_of_missing_values * 100



In [189]:
missing_percentage = missing_percentage.sort_values(ascending = False)
missing_percentage.head(15).round(1)

,0
"If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?",90.0
"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",85.8
Is your primary role within your company related to tech/IT?,81.6
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,80.0
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?",80.0
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?",80.0
"If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",80.0
Do you know local or online resources to seek help for a mental health disorder?,80.0
Do you believe your productivity is ever affected by a mental health issue?,80.0
"If maybe, what condition(s) do you believe you have?",77.5


The table that we get after runnign above shows that columns with very high missing values are mostly caused by the survey's conditional structure (for example, questions only shown to self-employed respondents). Since these are missing due to the survey's nature, they will be dropped or filtered later on.

In [190]:
print(f"Column with more than 50% missing values: {(missing_percentage > 50).sum()}")
print(f"Column with no missing values: {(missing_percentage == 0).sum()}")


Column with more than 50% missing values: 13
Column with no missing values: 19


# **Gender**

In [191]:
gender = df["What is your gender?"]
unique_genders = gender.unique()
# print(unique_genders)
print(f"Number of unique genders: {gender.nunique()}")
print(f"Missing values: {gender.isnull().sum()}")
gender.value_counts().head(20)


Number of unique genders: 70
Missing values: 3


,count
What is your gender?,
Male,610
male,249
Female,153
female,95
M,86
m,79
F,38
f,23
Male,11


The output shows that the gender column contains 70 different answer and 3 missing values, although many of them mean the same thing. For example, "Male","male","M" all refer to male and some even contain hidden spaces.


# **AGE**

In [192]:
age = df["What is your age?"]
age.describe()


,What is your age?
count,1433.000000
mean,34.286113
std,11.290931
min,3.000000
25%,28.000000
50%,33.000000
75%,39.000000
max,323.000000


In [193]:
unrealistic_ages = age[(age < 16) | (age>75)]
print(f"Number of unrealistic ages : {len(unrealistic_ages)}")
print(unrealistic_ages)

Number of unrealistic ages : 4
372     99
564    323
656      3
808     15
Name: What is your age?, dtype: int64


The age column has no missing values and the median age is 33 with most common age between 28 and 39. A few valus are unrealistic. They are most likely errors or typos or too young for regular employment. It is also very unlikely that somebody below the age of 16 would work in a tech company while 17 year olds could plausibly be interns or work voluntarily.

# Data Cleaning

In [194]:
self_employed = df['Are you self-employed?']
self_employed.value_counts()

,count
Are you self-employed?,
0,1146
1,287


In [195]:
is_employee = df['Are you self-employed?'] == 0
df_clean = df[is_employee].copy()
print(f"Rows before: {len(df)}")
print(f"Rows after: {len(df_clean)}")
print(f"self-employed: {len(df)-len(df_clean)}")


Rows before: 1433
Rows after: 1146
self-employed: 287


The dataset is filtered to employees only because the HR programme targets only the company's employees. Self-employed respondents could also not answer the employer-related questions, which caused many of the missing values. THe dataset is reduced from 1433 to 1146 respondents

In [196]:
missing_values = df_clean.isnull()
mean_of_missing_values = missing_values.mean()
missing_percentage = mean_of_missing_values * 100
# print(missing_percentage)
missing_percentage = missing_percentage.sort_values(ascending=False)
# print(missinng_percentage.head(15).round(1))
empty_columns = missing_percentage[missing_percentage == 100]
print(f"Columns that are now 100% empty: {len(empty_columns)}")
missing_percentage[missing_percentage>0].round(1)


Columns that are now 100% empty: 8


,0
Do you believe your productivity is ever affected by a mental health issue?,100.0
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?",100.0
"If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",100.0
"If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?",100.0
"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?",100.0
Do you know local or online resources to seek help for a mental health disorder?,100.0
Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,100.0
"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",100.0
"If maybe, what condition(s) do you believe you have?",78.2
Is your primary role within your company related to tech/IT?,77.1


After filtering the data, 8 columns became 100 empty since these questions were only shown to self employed respondents and contain no information for this analysis and will be dropped. Other columns still have missing values, mostly due to follow-up questions, US-only questions and some optional free test answers.

In [197]:
columns_to_drop = empty_columns.index
df_clean = df_clean.drop(columns=columns_to_drop)
print(f"Columns before: {df.shape[1]}")
print(f"Columns after: {df_clean.shape[1]}")

Columns before: 63
Columns after: 55


In [198]:
# df_clean.head()
other_columns_to_drop = [
    "Are you self-employed?",
    "Is your primary role within your company related to tech/IT?",
    "Why or why not?",
    "Why or why not?.1",
    "If maybe, what condition(s) do you believe you have?",
    "If yes, what condition(s) have you been diagnosed with?",
    "If so, what condition(s) were you diagnosed with?",
    "Have your observations of how another individual who discussed a mental health disorder made you less likely to reveal a mental health issue yourself in your current workplace?",
    "What US state or territory do you live in?",
    "What US state or territory do you work in?",
    "What country do you live in?"
    ]
print(f"Columns before: {df_clean.shape[1]}")
df_clean = df_clean.drop(columns=other_columns_to_drop)
print(f"Columns after: {df_clean.shape[1]}")


Columns before: 55
Columns after: 44


# Fixing Genders

In [199]:
gender_column = "What is your gender?"
df_clean[gender_column] = df_clean[gender_column].str.strip().str.lower()
print(f"Different answers after: {df_clean[gender_column].nunique()}")
df_clean[gender_column].value_counts()


Different answers after: 47


,count
What is your gender?,
male,707
female,209
m,125
f,44
woman,6
non-binary,4
man,3
cis male,3
agender,2


In [200]:
male_answers = ["male","m","cis male","male (cis)","male.","mail","dude","cisdude",
                "cis man","man", "sex is male","male (trans, ftm)"]
female_answers = ["female","f","woman","cis female","cisgender female","cis-woman",
                  "female/woman","fem","i identify as  female.","female (props for making this a freeform field, though)",
                  "transgender woman", "transitioned, m2f"]
def group_gender(answer):
  if pd.isnull(answer):
    return "unknown"
  elif answer in male_answers:
    return "male"
  elif answer in female_answers:
    return "female"
  else:
    return "other"

df_clean[gender_column] = df_clean[gender_column].apply(group_gender)
df_clean[gender_column].value_counts()

,count
What is your gender?,
male,847
female,267
other,29
unknown,3


The gender column orignally contained 70 different answers. First, extra spaces were removed and all answers were converted to lower case. After performing this step the number of different answer reduced to 47 then were grouped into four categories.

# Fixing age

In [201]:
age_column = "What is your age?"
realistic_age = (df_clean[age_column]>=16) & (df_clean[age_column]<=75)
unrealistic_rows = df_clean[~realistic_age]
print(f"Removed ages: {sorted(unrealistic_rows[age_column])}")
print(f"Rows before:{len(df_clean)}")
df_clean = df_clean[realistic_age].copy()
print(f"Rows after: {len(df_clean)}")

Removed ages: [3, 99, 323]
Rows before:1146
Rows after: 1143


unrealistic ages were removed

In [202]:
missing_count = df_clean.isnull().sum()
missing_count = missing_count[missing_count>0]
print(f"Columns with missing values: {len(missing_count)}")
missing_count

Columns with missing values: 13


,0
Do you know the options for mental health care available under your employer-provided coverage?,133
Have your previous employers provided mental health benefits?,131
Were you aware of the options for mental health care provided by your previous employers?,131
Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,131
Did your previous employers provide resources to learn more about mental health issues and how to seek help?,131
Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,131
Do you think that discussing a mental health disorder with previous employers would have negative consequences?,131
Do you think that discussing a physical health issue with previous employers would have negative consequences?,131
Would you have been willing to discuss a mental health issue with your previous co-workers?,131
Would you have been willing to discuss a mental health issue with your direct supervisor(s)?,131


In [203]:
previous_employer_col = "Do you have previous employers?"
df_clean[previous_employer_col].value_counts()

,count
Do you have previous employers?,
1,1012
0,131


The missinv alue check shows that 13 columns stil have gaps. 11 of them have exactly 131 missing values each, and all of these are questions about previous employers. The second part shows exactly that 131 respondents have no previous employers

In [204]:
previous_employer_col = missing_count[missing_count == 131].index
df_clean[previous_employer_col] = df_clean[previous_employer_col].fillna("No previous employer")
print(f"Columnns filled: {len(previous_employer_col)}")
print(f"Missing values left in these columns: {df_clean[previous_employer_col].isnull().sum().sum()}")


Columnns filled: 11
Missing values left in these columns: 0


The 11 previous employer columns were filled with the category "No previous employer". This keeps the information that these respondents had no previous employer

In [205]:
options_col = "Do you know the options for mental health care available under your employer-provided coverage?"
benefits_col = "Does your employer provide mental health benefits as part of healthcare coverage?"
options_missing = df_clean[options_col].isnull()
df_clean.loc[options_missing, benefits_col].value_counts()


,count
Does your employer provide mental health benefits as part of healthcare coverage?,
No,72
Not eligible for coverage / N/A,55
I don't know,6


In [206]:
df_clean[options_col] = df_clean[options_col].fillna("Not applicable")
print(f"Missing values left: {df_clean[options_col].isnull().sum()}")
df_clean[options_col].value_counts()

Missing values left: 0


,count
Do you know the options for mental health care available under your employer-provided coverage?,
No,352
I am not sure,351
Yes,307
Not applicable,133


In [207]:
unsupportive_col = "Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?"
print(df_clean[unsupportive_col].value_counts(dropna=False))
df_clean[unsupportive_col] = df_clean[unsupportive_col].fillna("Not answered")
print(f"Missing values left: {df_clean.isnull().sum().sum()}")

Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?
No                    491
Maybe/Not sure        276
Yes, I observed       193
Yes, I experienced    131
NaN                    52
Name: count, dtype: int64
Missing values left: 0


The question about knowing mental health care options had 131 missng values and since the follow-up questions did not apply to them, the gaps were filled with "Not applicable". The question about observing an unsupportive response had 52 missing values. This question was skilled by the respondents and the gaps were filled with "Not answered". To avoid inventing new information seperate categories were used instead. The dataset contains no missing values now.

In [208]:
short_names = ["company_size", "tech_company", "mh_benefits", "knows_care_options",
    "employer_discussed_mh", "employer_mh_resources", "anonymity_protected", "ease_medical_leave",
    "mh_negative_consequences", "ph_negative_consequences", "comfortable_coworkers", "comfortable_supervisor",
    "employer_takes_mh_seriously", "observed_consequences_coworkers", "has_previous_employer",
    "prev_mh_benefits", "prev_aware_options", "prev_employer_discussed_mh", "prev_mh_resources",
    "prev_anonymity_protected", "prev_mh_negative_consequences", "prev_ph_negative_consequences",
    "prev_comfortable_coworkers", "prev_comfortable_supervisor", "prev_takes_mh_seriously",
    "prev_observed_consequences", "interview_physical", "interview_mental", "hurt_career",
    "coworkers_view_negatively", "share_with_family", "observed_unsupportive_response",
    "family_history", "past_disorder", "current_disorder", "diagnosed_professional",
    "sought_treatment", "interferes_treated", "interferes_untreated", "age", "gender",
    "country_work", "work_position", "remote_work",]
print(f"Number of columns: {df_clean.shape[1]}, number of new names: {len(short_names)}")
col_lookup = pd.DataFrame({"short_name": short_names, "orignal questions": df_clean.columns})
df_clean.columns = short_names
df_clean.head()

Number of columns: 44, number of new names: 44


,company_size,tech_company,mh_benefits,knows_care_options,employer_discussed_mh,employer_mh_resources,anonymity_protected,ease_medical_leave,mh_negative_consequences,ph_negative_consequences,...,current_disorder,diagnosed_professional,sought_treatment,interferes_treated,interferes_untreated,age,gender,country_work,work_position,remote_work
0,26-100,1.0,Not eligible for coverage / N/A,Not applicable,No,No,I don't know,Very easy,No,No,...,No,Yes,0,Not applicable to me,Not applicable to me,39,male,United Kingdom,Back-end Developer,Sometimes
1,6-25,1.0,No,Yes,Yes,Yes,Yes,Somewhat easy,No,No,...,Yes,Yes,1,Rarely,Sometimes,29,male,United States of America,Back-end Developer|Front-end Developer,Never
2,6-25,1.0,No,Not applicable,No,No,I don't know,Neither easy nor difficult,Maybe,No,...,No,No,1,Not applicable to me,Not applicable to me,38,male,United Kingdom,Back-end Developer,Always
4,6-25,0.0,Yes,Yes,No,No,No,Neither easy nor difficult,Yes,Maybe,...,Yes,Yes,1,Sometimes,Sometimes,43,female,United States of America,Executive Leadership|Supervisor/Team Lead|Dev ...,Sometimes
5,More than 1000,1.0,Yes,I am not sure,No,Yes,Yes,Somewhat easy,Yes,Yes,...,Yes,No,1,Not applicable to me,Often,42,male,United Kingdom,DevOps/SysAdmin|Support|Back-end Developer|Fro...,Sometimes


The 44 columns were renamed from full survey questions to short names to make the following steps and plots readable.

In [209]:
print(f"Final shape: {df_clean.shape}")
print(f"Final missing values: {df_clean.isnull().sum().sum()}")

Final shape: (1143, 44)
Final missing values: 0


# Summary
The dataset from reduced from 1433 respondents and 63 columns to 1143 employyes and 44 columns. Self-employed respondents were removed alognside empty and unusable columns were dropped, gender answers categorized into four categories, unrealistic age were removed and al remaining missing values were filled with meaningfull categories. The data is now clean and ready for encoding.